# Lesson 3: Building an Agent Reasoning Loop

## Setup

In [ ]:
!pip install groq

In [ ]:
from groq import Groq
import os
from dotenv import load_dotenv

load_dotenv()

Groq_api_key = os.environ.get("GROQ_API_KEY")  # ← get the value, don't hardcode the name

client = Groq(
    api_key=Groq_api_key
)

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

key = os.environ.get("GROQ_API_KEY")
print(key)  # prints first 10 chars safely

In [ ]:
import nest_asyncio
nest_asyncio.apply()

## Load the data
To download this paper, below is the needed code:

#!wget "https://openreview.net/pdf?id=VtmBAGCN7o" -O metagpt.pdf

## Setup the Query Tools

In [ ]:
!pip install llama-index-llms-groq llama-index-embeddings-huggingface

In [ ]:
from llama_index.core import Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.groq import Groq

Settings.llm = Groq(
    model="llama-3.1-8b-instant",
    api_key=os.environ["GROQ_API_KEY"]
)

Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5"
)

print(Settings.embed_model)

In [ ]:
from utils import get_doc_tools

vector_tool, summary_tool = get_doc_tools("metagpt.pdf", "metagpt")

## Setup Function Calling Agent

In [ ]:
from llama_index.llms.groq import Groq
import os

llm = Groq(
    model="llama-3.1-8b-instant",
    api_key=os.environ["GROQ_API_KEY"]
)

In [ ]:
from llama_index.core.agent import ReActAgent

agent = ReActAgent(
    tools=[vector_tool, summary_tool],
    llm=llm,
    verbose=True,
    streaming=False
)

In [ ]:
response = await agent.run(
    "Tell me about the agent roles in MetaGPT, "
    "and then how they communicate with each other."
)
print(str(response))

## Lower-Level: Debuggability and Control

In [ ]:
from llama_index.core.agent import ReActAgent

agent = ReActAgent(
    tools=[vector_tool, summary_tool],
    llm=llm,
    verbose=True,
    streaming=False
)

In [ ]:
from llama_index.core.agent.workflow import FunctionAgent

agent = FunctionAgent(
    tools=[vector_tool, summary_tool],
    llm=llm,
    verbose=True,
)

response = await agent.run(
    "Tell me about the agent roles in MetaGPT, "
    "and then how they communicate with each other."
)
print(str(response))

In [ ]:
from llama_index.core.memory import ChatMemoryBuffer

memory = ChatMemoryBuffer.from_defaults(token_limit=40000)

agent = FunctionAgent(
    tools=[vector_tool, summary_tool],
    llm=llm,
    verbose=True,
    memory=memory,
)

# First question
response1 = await agent.run("Tell me about the agent roles in MetaGPT.")
print(str(response1))

# Follow-up — memory keeps context
response2 = await agent.run("What about how agents share information?")
print(str(response2))